In [7]:
import numpy as np
# Ввод матрицы платежей
C = np.array([[12, 9, 18], 
              [15, 22, 5], 
              [16, 3, 12]])

# Метод обратной матрицы (аналитическое решение)
# Вычисляем определитель и обратную матрицу.
# Используем единичный вектор u = [1, 1, 1].
# Находим оптимальные стратегии и цену игры.

# Функция для вычисления определителя матрицы
def determinant_matrix(mat):
    n = len(mat) # размерность квадратной матрицы
    if n == 1:
        return mat[0][0]
    if n == 2:
        return mat[0][0] * mat[1][1] - mat[0][1] * mat[1][0]
    
    det = 0
    for j in range(n): # перебираем элементы первой строки
        sub_mat = np.delete(np.delete(mat, 0, axis=0), j, axis=1) # минор (подматрица, которая остаётся после удаления i-й строки и j-го столбца)
        det += ((-1) ** j) * mat[0][j] * determinant_matrix(sub_mat) # ищем определитель: (-1)^j * c_0j * det(sub_mat)

    return det

# Функция для вычисления обратной матрицы
def inverse_matrix(mat):
    n = len(mat)
    det = determinant_matrix(mat)

    if det == 0:
        raise ValueError("Матрица вырожденная, обратной матрицы не существует.") # генерация исключения
    
    # Создаём матрицу алгебраических дополнений
    adjugate = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            sub_mat = np.delete(np.delete(mat, i, axis=0), j, axis=1) 
            adjugate[j][i] = ((-1) ** (i + j)) * determinant_matrix(sub_mat)
    
    return adjugate / det

C_inv = inverse_matrix(C) #обратная матрица
# Аналитический метод (обратная матрица)
# Вычисление оптимальных стратегий
u = np.ones(len(C))
v = 1 / np.dot(np.dot(u, C_inv), u)
p = np.dot(C_inv, u) * v
q = np.dot(u, C_inv) * v


print(f"Цена игры (v): {v}")
print(f"Оптимальная стратегия игрока A: {p}")
print(f"Оптимальная стратегия игрока B: {q}")


Цена игры (v): 13.223076923076922
Оптимальная стратегия игрока A: [0.6        0.13076923 0.26923077]
Оптимальная стратегия игрока B: [0.60769231 0.34615385 0.04615385]


In [ ]:
# Метод Брауна–Робинсона (итерационный алгоритм)
# Итерационно приближаем оптимальные стратегии, анализируя выигрыши игроков.

def braun_robinson(A, N=100, epsilon=0.1):
    n, m = A.shape
    p_freq = np.zeros(n)
    q_freq = np.zeros(m)
    
    i, j = 0, 0
    vmin, vmax = -np.inf, np.inf
    
    for k in range(1, N + 1):
        p_freq[i] += 1
        q_freq[j] += 1
        
        row_payoff = A[:, j]
        col_payoff = A[i, :]
        
        i = np.argmax(row_payoff)
        j = np.argmin(col_payoff)
        
        vmin = max(vmin, np.min(col_payoff) / k)
        vmax = min(vmax, np.max(row_payoff) / k)
        
        if abs(vmax - vmin) < epsilon:
            break

    p = p_freq / k
    q = q_freq / k
    v = (vmin + vmax) / 2

    return v, p, q

v_br, p_br, q_br = braun_robinson(A)
print(f"\nМетод Брауна–Робинсона:\nЦена игры: {v_br:.4f}")
print(f"Оптимальная стратегия игрока A: {p_br}")
print(f"Оптимальная стратегия игрока B: {q_br}")

Метод Брауна–Робинсона:
Цена игры: 4.59
Стратегия A: [0.49 0.5  0.01]
Стратегия B: [0.01 0.5  0.49]
